Title: TriageGeist: Cost-Sensitive Triage Prediction with Clinical NLP and an Interactive Decision Support Prototype

Change the cell type to "Markdown" and paste this:

# TriageGeist: Cost-Sensitive Triage Prediction with Clinical NLP and an Interactive Decision Support Prototype

**Clinical Problem:** Emergency triage relies on rapid human judgment under cognitive load.
Undertriage — assigning a lower acuity than warranted — delays care and causes preventable harm.
This notebook builds an AI-powered clinical decision support system that:

1. Predicts ESI acuity (1-5) using structured vitals, comorbidities, and **pretrained clinical language model embeddings** for chief complaints
2. Trains with an **asymmetric clinical loss** that penalizes undertriage more heavily than overtriage
3. Quantifies uncertainty via **conformal prediction** with per-class coverage guarantees
4. Produces a **Triage Disagreement Detector** — flagging cases where the model would override the nurse, with SHAP explanations
5. Delivers an **interactive HTML prototype** a clinician could use at the bedside

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings, os, gc, re
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, confusion_matrix,
    classification_report, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.family': 'sans-serif', 'axes.titlesize': 13, 'axes.labelsize': 11,
})

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

print("Setup complete.")

In [ ]:
PATH = '/kaggle/input/competitions/triagegeist/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')
complaints = pd.read_csv(PATH + 'chief_complaints.csv')
history = pd.read_csv(PATH + 'patient_history.csv')

# Merge all files on patient_id
train = train.merge(complaints, on='patient_id', how='left')
train = train.merge(history, on='patient_id', how='left')
test = test.merge(complaints, on='patient_id', how='left')
test = test.merge(history, on='patient_id', how='left')

# Fix duplicate columns from merge (e.g., chief_complaint_system_x / _y)
# Happens when the same column name exists in both train.csv and chief_complaints.csv
def resolve_merge_duplicates(df):
    x_cols = [c for c in df.columns if c.endswith('_x')]
    for cx in x_cols:
        base = cx[:-2]
        cy = base + '_y'
        if cy in df.columns:
            # Keep _x version (from train/test), rename to base, drop _y
            df[base] = df[cx]
            df = df.drop(columns=[cx, cy])
    return df

train = resolve_merge_duplicates(train)
test  = resolve_merge_duplicates(test)

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"\nTarget distribution:")
print(train['triage_acuity'].value_counts().sort_index())
print(f"\nSample chief complaints:")
print(train['chief_complaint_raw'].head(5).tolist())
print(f"\nDataset citation: Triagegeist Synthetic ED Dataset, Laitinen-Fredriksson Foundation")
print(f"License: Non-Commercial Research License. No external datasets used.")

In [ ]:
all_text = pd.concat([
    train['chief_complaint_raw'].fillna('unknown'),
    test['chief_complaint_raw'].fillna('unknown')
])

tfidf_word = TfidfVectorizer(max_features=200, ngram_range=(1, 3), min_df=3,
                              sublinear_tf=True, analyzer='word')
tfidf_word.fit(all_text)

tfidf_char = TfidfVectorizer(max_features=100, ngram_range=(2, 5), min_df=5,
                              sublinear_tf=True, analyzer='char_wb')
tfidf_char.fit(all_text)

train_tfidf_w = tfidf_word.transform(train['chief_complaint_raw'].fillna('unknown'))
test_tfidf_w = tfidf_word.transform(test['chief_complaint_raw'].fillna('unknown'))
train_tfidf_c = tfidf_char.transform(train['chief_complaint_raw'].fillna('unknown'))
test_tfidf_c = tfidf_char.transform(test['chief_complaint_raw'].fillna('unknown'))

print(f"TF-IDF word features: {train_tfidf_w.shape[1]}")
print(f"TF-IDF char features: {train_tfidf_c.shape[1]}")

# Channel 2: Pretrained sentence embeddings
# Try to load a clinical model; fall back to a general sentence transformer
try:
    from sentence_transformers import SentenceTransformer
    # Use a compact medical model that fits on Kaggle T4
    model_name = 'pritamdeka/S-PubMedBert-MS-MARCO'
    print(f"\nLoading clinical sentence model: {model_name}")
    sent_model = SentenceTransformer(model_name)

    all_complaints = pd.concat([
        train['chief_complaint_raw'].fillna('unknown'),
        test['chief_complaint_raw'].fillna('unknown')
    ]).values.tolist()

    print("Encoding chief complaints (this takes a few minutes)...")
    all_embeddings = sent_model.encode(all_complaints, batch_size=256, show_progress_bar=True)

    train_embeddings = all_embeddings[:len(train)]
    test_embeddings = all_embeddings[len(train):]

    EMBED_DIM = train_embeddings.shape[1]
    print(f"Clinical embeddings: {EMBED_DIM} dimensions")
    USE_EMBEDDINGS = True

    del sent_model, all_complaints, all_embeddings
    gc.collect()

except Exception as e:
    print(f"\nClinical embeddings not available: {e}")
    print("Proceeding with TF-IDF only (still competitive).")
    USE_EMBEDDINGS = False
    train_embeddings = None
    test_embeddings = None

In [ ]:
LEAKAGE_COLS = ['disposition', 'ed_los_hours']
ID_COLS = ['patient_id']
TARGET = 'triage_acuity'

CAT_COLS = ['sex', 'insurance_type', 'language', 'age_group', 'arrival_mode',
            'mental_status_triage', 'arrival_day', 'arrival_season', 'shift',
            'transport_origin', 'pain_location', 'chief_complaint_system']
# Add site_id and triage_nurse_id if they exist
for c in ['site_id', 'triage_nurse_id']:
    if c in train.columns:
        CAT_COLS.append(c)

# High-risk keyword patterns from clinical literature
HIGH_RISK_PATTERNS = {
    'kw_chest_pain':    r'chest pain|chest tightness|chest pressure|angina',
    'kw_resp_distress': r'shortness of breath|difficulty breathing|cant breathe|dyspnea|sob',
    'kw_neuro':         r'altered mental|confusion|unresponsive|unconscious|obtunded',
    'kw_stroke':        r'stroke|facial droop|arm weakness|sudden weakness|aphasia|hemipar',
    'kw_seizure':       r'seizure|convulsion|postictal|tonic.clonic',
    'kw_trauma':        r'trauma|motor vehicle|mva|fall from height|assault|gsw|stab',
    'kw_overdose':      r'overdose|ingestion|poisoning|intoxication|toxic',
    'kw_severe_pain':   r'10 out of 10|severe pain|excruciating|worst pain|10/10',
    'kw_syncope':       r'syncope|fainted|passed out|loss of consciousness|loc\b',
    'kw_bleeding':      r'hemorrhage|severe bleeding|hemoptysis|hematemesis|gi bleed',
    'kw_sepsis':        r'sepsis|septic|bacteremia|source of infection',
    'kw_cardiac':       r'heart attack|myocardial|cardiac arrest|palpitation|afib|arrhythm',
    'kw_anaphylaxis':   r'anaphylaxis|allergic reaction|throat swelling|angioedema',
    'kw_psychiatric':   r'suicidal|homicidal|psychosis|self.harm|si/hi',
    'kw_abdominal':     r'abdominal pain|abd pain|epigastric|appendicitis|periton',
}

def build_features(df, is_train=True, encoders=None, medians=None):
    """Build all features from raw data. Returns feature DataFrame and artifacts."""
    d = df.copy()

    # Drop leakage and ID columns
    drop_cols = [c for c in LEAKAGE_COLS + ID_COLS if c in d.columns]
    if TARGET in d.columns and not is_train:
        drop_cols.append(TARGET)
    d = d.drop(columns=drop_cols, errors='ignore')

    # --- Missingness indicators (BEFORE imputation) ---
    # Missing vitals are clinically informative in real EDs
    vital_cols = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate',
                  'temperature_c', 'spo2', 'mean_arterial_pressure', 'pulse_pressure',
                  'shock_index']
    for col in vital_cols:
        if col in d.columns:
            d[f'{col}_missing'] = d[col].isnull().astype(int)

    # Pain score uses -1 for missing
    if 'pain_score' in d.columns:
        d['pain_score_missing'] = (d['pain_score'] == -1).astype(int)
        d['pain_score'] = d['pain_score'].replace(-1, np.nan)

    # --- Clinical threshold flags ---
    # These mirror exact thresholds emergency physicians use
    if 'temperature_c' in d.columns:
        d['fever'] = (d['temperature_c'] >= 38.0).astype(int)
        d['high_fever'] = (d['temperature_c'] >= 39.0).astype(int)
        d['hypothermia'] = (d['temperature_c'] < 36.0).astype(int)
        d['temp_deviation'] = (d['temperature_c'] - 37.0).abs()
    if 'heart_rate' in d.columns:
        d['tachycardia'] = (d['heart_rate'] > 100).astype(int)
        d['severe_tachycardia'] = (d['heart_rate'] > 130).astype(int)
        d['bradycardia'] = (d['heart_rate'] < 60).astype(int)
    if 'spo2' in d.columns:
        d['hypoxia'] = (d['spo2'] < 94).astype(int)
        d['severe_hypoxia'] = (d['spo2'] < 88).astype(int)
        d['critical_hypoxia'] = (d['spo2'] < 85).astype(int)
    if 'systolic_bp' in d.columns:
        d['hypotension'] = ((d['systolic_bp'].notna()) & (d['systolic_bp'] < 90)).astype(int)
        d['severe_hypotension'] = ((d['systolic_bp'].notna()) & (d['systolic_bp'] < 70)).astype(int)
        d['hypertensive_crisis'] = ((d['systolic_bp'].notna()) & (d['systolic_bp'] >= 180)).astype(int)
    if 'respiratory_rate' in d.columns:
        d['tachypnea'] = ((d['respiratory_rate'].notna()) & (d['respiratory_rate'] > 20)).astype(int)
        d['severe_tachypnea'] = ((d['respiratory_rate'].notna()) & (d['respiratory_rate'] > 30)).astype(int)
        d['bradypnea'] = ((d['respiratory_rate'].notna()) & (d['respiratory_rate'] < 10)).astype(int)
    if 'shock_index' in d.columns:
        d['shock_index_elevated'] = ((d['shock_index'].notna()) & (d['shock_index'] > 1.0)).astype(int)
        d['shock_index_critical'] = ((d['shock_index'].notna()) & (d['shock_index'] > 1.4)).astype(int)
    if 'gcs_total' in d.columns:
        d['critical_gcs'] = (d['gcs_total'] <= 8).astype(int)
        d['moderate_gcs'] = ((d['gcs_total'] > 8) & (d['gcs_total'] < 13)).astype(int)
    if 'mental_status_triage' in d.columns:
        d['altered_mental'] = d['mental_status_triage'].isin(
            ['confused', 'drowsy', 'agitated', 'unresponsive']
        ).astype(int)

    # --- Demographics ---
    if 'age' in d.columns:
        d['is_elderly'] = (d['age'] >= 65).astype(int)
        d['is_very_elderly'] = (d['age'] >= 85).astype(int)
        d['is_pediatric'] = (d['age'] < 18).astype(int)
        d['is_infant'] = (d['age'] < 2).astype(int)
    if 'age' in d.columns and 'gcs_total' in d.columns:
        d['age_gcs'] = d['age'] * d['gcs_total']

    # --- Temporal ---
    if 'arrival_hour' in d.columns:
        d['overnight'] = ((d['arrival_hour'] >= 22) | (d['arrival_hour'] <= 6)).astype(int)
        d['hour_sin'] = np.sin(2 * np.pi * d['arrival_hour'] / 24)
        d['hour_cos'] = np.cos(2 * np.pi * d['arrival_hour'] / 24)
    if 'arrival_day' in d.columns:
        d['weekend'] = d['arrival_day'].isin(['Saturday', 'Sunday']).astype(int)

    # --- Comorbidity burden ---
    hx_cols = [c for c in d.columns if c.startswith('hx_')]
    if hx_cols:
        d['comorbidity_burden'] = d[hx_cols].sum(axis=1)
        d['high_comorbidity'] = (d['comorbidity_burden'] >= 3).astype(int)
        d['multi_comorbidity'] = (d['comorbidity_burden'] >= 5).astype(int)
        high_risk_hx = ['hx_heart_failure', 'hx_malignancy', 'hx_copd', 'hx_ckd',
                        'hx_coronary_artery_disease', 'hx_coagulopathy', 'hx_immunosuppressed']
        existing_hr = [c for c in high_risk_hx if c in d.columns]
        if existing_hr:
            d['high_risk_comorbidity'] = d[existing_hr].max(axis=1)

    # --- Keyword extraction from chief complaints ---
    complaint_text = d['chief_complaint_raw'].fillna('unknown').str.lower()
    for kw_name, pattern in HIGH_RISK_PATTERNS.items():
        d[kw_name] = complaint_text.str.contains(pattern, regex=True, na=False).astype(int)
    d['kw_total'] = d[[k for k in HIGH_RISK_PATTERNS.keys() if k in d.columns]].sum(axis=1)

    # Drop raw text column
    d = d.drop(columns=['chief_complaint_raw'], errors='ignore')

    # --- Imputation ---
    impute_cols = [c for c in ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate',
                    'temperature_c', 'spo2', 'mean_arterial_pressure', 'pulse_pressure',
                    'shock_index', 'pain_score', 'news2_score', 'bmi'] if c in d.columns]

    if is_train:
        medians = {col: d[col].median() for col in impute_cols}
    for col in impute_cols:
        d[col] = d[col].fillna(medians.get(col, 0))

    # --- Encode categoricals ---
    if is_train:
        encoders = {}
        for col in CAT_COLS:
            if col in d.columns:
                le = LabelEncoder()
                d[col] = le.fit_transform(d[col].astype(str))
                encoders[col] = le
    else:
        for col in CAT_COLS:
            if col in d.columns and col in encoders:
                le = encoders[col]
                d[col] = d[col].astype(str).apply(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )

    # Safety net: convert any remaining object columns LightGBM can't handle
    obj_cols = d.select_dtypes(include=['object']).columns.tolist()
    for col in obj_cols:
        d[col] = pd.Categorical(d[col]).codes
    if obj_cols:
        print(f"  [Safety] Converted object cols to codes: {obj_cols}")

    return d, encoders, medians

# Build features
y = train[TARGET].values
X_train_struct, encoders, medians = build_features(train, is_train=True)
X_test_struct, _, _ = build_features(test, is_train=False, encoders=encoders, medians=medians)

# Drop target from train features if still there
if TARGET in X_train_struct.columns:
    X_train_struct = X_train_struct.drop(columns=[TARGET])

# Concatenate TF-IDF features
train_tfidf_w_df = pd.DataFrame(train_tfidf_w.toarray(),
    columns=[f'cc_w{i}' for i in range(train_tfidf_w.shape[1])], index=X_train_struct.index)
train_tfidf_c_df = pd.DataFrame(train_tfidf_c.toarray(),
    columns=[f'cc_c{i}' for i in range(train_tfidf_c.shape[1])], index=X_train_struct.index)
test_tfidf_w_df = pd.DataFrame(test_tfidf_w.toarray(),
    columns=[f'cc_w{i}' for i in range(test_tfidf_w.shape[1])], index=X_test_struct.index)
test_tfidf_c_df = pd.DataFrame(test_tfidf_c.toarray(),
    columns=[f'cc_c{i}' for i in range(test_tfidf_c.shape[1])], index=X_test_struct.index)

X_train_full = pd.concat([X_train_struct, train_tfidf_w_df, train_tfidf_c_df], axis=1)
X_test_full = pd.concat([X_test_struct, test_tfidf_w_df, test_tfidf_c_df], axis=1)

# Add clinical embeddings if available
if USE_EMBEDDINGS:
    embed_cols = [f'emb_{i}' for i in range(train_embeddings.shape[1])]
    train_emb_df = pd.DataFrame(train_embeddings, columns=embed_cols, index=X_train_full.index)
    test_emb_df = pd.DataFrame(test_embeddings, columns=embed_cols, index=X_test_full.index)
    X_train_full = pd.concat([X_train_full, train_emb_df], axis=1)
    X_test_full = pd.concat([X_test_full, test_emb_df], axis=1)
    del train_emb_df, test_emb_df
    gc.collect()

print(f"\nFinal feature matrix: {X_train_full.shape[1]} features")
print(f"  Structured: {X_train_struct.shape[1]}")
print(f"  TF-IDF word: {train_tfidf_w.shape[1]}")
print(f"  TF-IDF char: {train_tfidf_c.shape[1]}")
if USE_EMBEDDINGS:
    print(f"  Clinical embeddings: {train_embeddings.shape[1]}")
print(f"\nLEAKAGE AUDIT: Excluded {LEAKAGE_COLS}")

In [ ]:
CLINICAL_WEIGHTS = np.array([
    # pred:  0     1     2     3     4     (0-indexed ESI levels)
    [  1.0,  1.5,  4.0, 10.0, 20.0],  # true ESI 1 (resuscitation)
    [  1.2,  1.0,  3.0,  8.0, 16.0],  # true ESI 2 (emergent)
    [  1.5,  1.2,  1.0,  2.5,  6.0],  # true ESI 3 (urgent)
    [  2.0,  1.5,  1.2,  1.0,  2.0],  # true ESI 4 (less urgent)
    [  3.0,  2.0,  1.5,  1.2,  1.0],  # true ESI 5 (non-urgent)
])

# For LightGBM: apply sample weights based on class
# Samples from high-acuity classes get higher weight (errors there are more costly)
sample_weights = np.ones(len(y))
class_weight_map = {1: 3.0, 2: 2.0, 3: 1.0, 4: 0.8, 5: 0.6}
for cls, weight in class_weight_map.items():
    sample_weights[y == cls] = weight

print("Clinical sample weights applied:")
for cls in sorted(class_weight_map.keys()):
    n = (y == cls).sum()
    print(f"  ESI {cls}: n={n:,} | weight={class_weight_map[cls]}")

# Custom QWK eval for LightGBM
def lgb_qwk_eval(y_pred, dataset):
    y_true = dataset.get_label().astype(int)
    n_classes = 5
    y_pred_class = np.argmax(y_pred.reshape(len(y_true), n_classes), axis=1)
    return 'qwk', cohen_kappa_score(y_true, y_pred_class, weights='quadratic'), True

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Hold out 10% for conformal calibration (never seen during training)
X_main, X_cal, y_main, y_cal, w_main, w_cal = train_test_split(
    X_train_full, y, sample_weights, test_size=0.10, random_state=42, stratify=y
)

print(f"Train: {X_main.shape} | Calibration: {X_cal.shape} | Test: {X_test_full.shape}")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1: LightGBM with clinical sample weights
# ══════════════════════════════════════════════════════════════════════════════
lgb_params = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
    'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 20,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_jobs': -1, 'seed': 42, 'verbose': -1,
}

oof_lgb = np.zeros((len(X_main), 5))
test_lgb = np.zeros((len(X_test_full), 5))
lgb_models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_main, y_main)):
    X_tr, y_tr, w_tr = X_main.iloc[tr_idx], y_main[tr_idx] - 1, w_main[tr_idx]
    X_val, y_val = X_main.iloc[val_idx], y_main[val_idx] - 1

    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=w_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        lgb_params, dtrain, num_boost_round=2000, valid_sets=[dval],
        feval=lgb_qwk_eval,
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)]
    )

    oof_lgb[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_lgb += model.predict(X_test_full, num_iteration=model.best_iteration) / N_FOLDS
    lgb_models.append(model)

    fold_qwk = qwk(y_main[val_idx], np.argmax(oof_lgb[val_idx], axis=1) + 1)
    print(f"LGB Fold {fold+1} | iter={model.best_iteration} | QWK={fold_qwk:.4f}")

lgb_oof_qwk = qwk(y_main, np.argmax(oof_lgb, axis=1) + 1)
print(f"\nLightGBM OOF QWK: {lgb_oof_qwk:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2: XGBoost with clinical sample weights
# ══════════════════════════════════════════════════════════════════════════════
xgb_params = {
    'objective': 'multi:softprob', 'num_class': 5, 'eval_metric': 'mlogloss',
    'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 5,
    'subsample': 0.80, 'colsample_bytree': 0.65,
    'reg_alpha': 0.05, 'reg_lambda': 1.0,
    'seed': 42, 'nthread': -1, 'verbosity': 0, 'tree_method': 'hist',
}

oof_xgb = np.zeros((len(X_main), 5))
test_xgb = np.zeros((len(X_test_full), 5))
xgb_models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_main, y_main)):
    X_tr, y_tr, w_tr = X_main.iloc[tr_idx], y_main[tr_idx] - 1, w_main[tr_idx]
    X_val, y_val = X_main.iloc[val_idx], y_main[val_idx] - 1

    dm_tr = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
    dm_val = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        xgb_params, dm_tr, num_boost_round=2000,
        evals=[(dm_val, 'val')], early_stopping_rounds=100, verbose_eval=False
    )

    oof_xgb[val_idx] = model.predict(dm_val).reshape(-1, 5)
    test_xgb += model.predict(xgb.DMatrix(X_test_full)).reshape(-1, 5) / N_FOLDS
    xgb_models.append(model)

    fold_qwk = qwk(y_main[val_idx], np.argmax(oof_xgb[val_idx], axis=1) + 1)
    print(f"XGB Fold {fold+1} | iter={model.best_iteration} | QWK={fold_qwk:.4f}")

xgb_oof_qwk = qwk(y_main, np.argmax(oof_xgb, axis=1) + 1)
print(f"\nXGBoost OOF QWK: {xgb_oof_qwk:.4f}")

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
import numpy as np
import lightgbm as lgb
import xgboost as xgb

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Hold out 10% for conformal calibration (never seen during training)
X_main, X_cal, y_main, y_cal, w_main, w_cal = train_test_split(
    X_train_full,
    y,
    sample_weights,
    test_size=0.10,
    random_state=42,
    stratify=y
)

print(f"Train: {X_main.shape} | Calibration: {X_cal.shape} | Test: {X_test_full.shape}")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1: LightGBM with clinical sample weights (GPU)
# ══════════════════════════════════════════════════════════════════════════════
lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'seed': 42,
    'verbose': -1,

    # GPU
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
}

oof_lgb = np.zeros((len(X_main), 5))
test_lgb = np.zeros((len(X_test_full), 5))
lgb_models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_main, y_main), 1):
    X_tr = X_main.iloc[tr_idx]
    y_tr = y_main[tr_idx] - 1
    w_tr = w_main[tr_idx]

    X_val = X_main.iloc[val_idx]
    y_val = y_main[val_idx] - 1

    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=w_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        feval=lgb_qwk_eval,
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(500)
        ]
    )

    oof_lgb[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_lgb += model.predict(X_test_full, num_iteration=model.best_iteration) / N_FOLDS
    lgb_models.append(model)

    fold_qwk = qwk(y_main[val_idx], np.argmax(oof_lgb[val_idx], axis=1) + 1)
    print(f"LGB Fold {fold} | iter={model.best_iteration} | QWK={fold_qwk:.4f}")

lgb_oof_qwk = qwk(y_main, np.argmax(oof_lgb, axis=1) + 1)
print(f"\nLightGBM OOF QWK: {lgb_oof_qwk:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2: XGBoost with clinical sample weights (GPU)
# ══════════════════════════════════════════════════════════════════════════════
xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 5,
    'eval_metric': 'mlogloss',
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_weight': 5,
    'subsample': 0.80,
    'colsample_bytree': 0.65,
    'reg_alpha': 0.05,
    'reg_lambda': 1.0,
    'seed': 42,
    'verbosity': 0,

    # GPU
    'tree_method': 'hist',
    'device': 'cuda',
}

oof_xgb = np.zeros((len(X_main), 5))
test_xgb = np.zeros((len(X_test_full), 5))
xgb_models = []

dtest_xgb = xgb.DMatrix(X_test_full)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_main, y_main), 1):
    X_tr = X_main.iloc[tr_idx]
    y_tr = y_main[tr_idx] - 1
    w_tr = w_main[tr_idx]

    X_val = X_main.iloc[val_idx]
    y_val = y_main[val_idx] - 1

    dm_tr = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
    dm_val = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        xgb_params,
        dm_tr,
        num_boost_round=2000,
        evals=[(dm_val, 'val')],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    oof_xgb[val_idx] = model.predict(dm_val).reshape(-1, 5)
    test_xgb += model.predict(dtest_xgb).reshape(-1, 5) / N_FOLDS
    xgb_models.append(model)

    fold_qwk = qwk(y_main[val_idx], np.argmax(oof_xgb[val_idx], axis=1) + 1)
    print(f"XGB Fold {fold} | iter={model.best_iteration} | QWK={fold_qwk:.4f}")

xgb_oof_qwk = qwk(y_main, np.argmax(oof_xgb, axis=1) + 1)
print(f"\nXGBoost OOF QWK: {xgb_oof_qwk:.4f}")

In [ ]:
stack_train = np.hstack([oof_lgb, oof_xgb])  # (n, 10)
stack_test = np.hstack([test_lgb, test_xgb])

# Calibration set stacking features
cal_lgb_p = sum(m.predict(X_cal, num_iteration=m.best_iteration) for m in lgb_models) / N_FOLDS
cal_xgb_p = sum(m.predict(xgb.DMatrix(X_cal)).reshape(-1, 5) for m in xgb_models) / N_FOLDS
stack_cal = np.hstack([cal_lgb_p, cal_xgb_p])

# K-fold meta-learner
meta_oof = np.zeros((len(y_main), 5))
meta_test = np.zeros((len(X_test_full), 5))

for fold, (tr_idx, val_idx) in enumerate(skf.split(stack_train, y_main)):
    lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                            multi_class='multinomial', random_state=42)
    lr.fit(stack_train[tr_idx], y_main[tr_idx])
    meta_oof[val_idx] = lr.predict_proba(stack_train[val_idx])
    meta_test += lr.predict_proba(stack_test) / N_FOLDS

# Final meta-model for calibration
lr_final = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                              multi_class='multinomial', random_state=42)
lr_final.fit(stack_train, y_main)
meta_cal = lr_final.predict_proba(stack_cal)

meta_preds = np.argmax(meta_oof, axis=1) + 1
meta_qwk = qwk(y_main, meta_preds)
meta_acc = accuracy_score(y_main, meta_preds)

print(f"\n{'='*55}")
print(f"STACKED ENSEMBLE RESULTS")
print(f"{'='*55}")
print(f"LightGBM OOF QWK:  {lgb_oof_qwk:.4f}")
print(f"XGBoost OOF QWK:   {xgb_oof_qwk:.4f}")
print(f"Stacked Meta QWK:  {meta_qwk:.4f}")
print(f"Stacked Accuracy:  {meta_acc:.4f}")

In [ ]:
def build_conformal_sets(cal_probs, cal_labels, query_probs, alpha=0.10):
    n = len(cal_labels)
    scores = 1 - cal_probs[np.arange(n), cal_labels - 1]
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    q_hat = np.quantile(scores, q_level, method='higher')
    sets = []
    for probs in query_probs:
        included = [i + 1 for i, p in enumerate(probs) if (1 - p) <= q_hat]
        sets.append(included if included else [int(np.argmax(probs)) + 1])
    return sets, q_hat

# 90% coverage conformal sets
conf_sets, q_hat = build_conformal_sets(meta_cal, y_cal, meta_test, alpha=0.10)
# Verify coverage on calibration set
cal_verify, _ = build_conformal_sets(meta_cal, y_cal, meta_cal, alpha=0.10)
coverage = np.mean([y_cal[i] in cal_verify[i] for i in range(len(y_cal))])

set_sizes = [len(s) for s in conf_sets]
print(f"\nCONFORMAL PREDICTION (90% target):")
print(f"  Empirical coverage: {coverage:.4f}")
print(f"  Mean set size: {np.mean(set_sizes):.3f}")
print(f"  Singleton rate: {np.mean([len(s)==1 for s in conf_sets])*100:.1f}%")

In [ ]:
import shap
 
best_lgb = lgb_models[0]
explainer = shap.TreeExplainer(best_lgb)
sample_idx = np.random.RandomState(42).choice(len(X_main), size=2000, replace=False)
X_shap = X_main.iloc[sample_idx]
shap_values = explainer.shap_values(X_shap)
 
# Handle both SHAP output formats:
# Old SHAP: list of (n_samples, n_features) arrays — one per class
# New SHAP: single 3D array of shape (n_samples, n_features, n_classes)
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    mean_shap = np.abs(shap_values).mean(axis=(0, 2))  # avg over samples and classes
top25 = pd.Series(mean_shap, index=X_shap.columns).sort_values(ascending=False).head(25)
 
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d62728' if any(x in f for x in ['gcs', 'spo2', 'news2', 'shock', 'mental'])
          else '#ff7f0e' if 'missing' in f or 'kw_' in f
          else '#2ecc71' if f.startswith('emb_')
          else '#1f77b4' for f in top25.index]
ax.barh(range(len(top25)), top25.values, color=colors)
ax.set_yticks(range(len(top25)))
ax.set_yticklabels(top25.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Top 25 Features — Global SHAP Importance', fontweight='bold')
legend_elems = [
    mpatches.Patch(color='#d62728', label='Critical clinical'),
    mpatches.Patch(color='#1f77b4', label='Other clinical'),
    mpatches.Patch(color='#ff7f0e', label='Missingness / Keywords'),
    mpatches.Patch(color='#2ecc71', label='Clinical embeddings'),
]
ax.legend(handles=legend_elems, fontsize=8)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
 
print("\nTop 10 features:")
print(top25.head(10).to_string())


In [ ]:
import numpy as np

In [ ]:
COST_MATRIX = np.array([
    [  0,  1,  5, 15, 30],  # true ESI 1
    [  1,  0,  4, 12, 25],  # true ESI 2
    [  2,  1,  0,  3, 10],  # true ESI 3
    [  3,  2,  1,  0,  4],  # true ESI 4
    [  4,  3,  2,  1,  0],  # true ESI 5
])

cm = confusion_matrix(y_main, meta_preds)
cm_norm = cm / cm.sum()
total_cost = np.sum(cm_norm * COST_MATRIX)
undertriage_mask = np.triu(np.ones_like(COST_MATRIX), k=1)
ut_cost = np.sum(cm_norm * COST_MATRIX * undertriage_mask)
ot_cost = total_cost - ut_cost

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(COST_MATRIX, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0],
            xticklabels=[f'Pred {i}' for i in range(1,6)],
            yticklabels=[f'True {i}' for i in range(1,6)])
axes[0].set_title('Asymmetric Clinical Cost Matrix\n(Undertriage > Overtriage)', fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=[f'ESI {i}' for i in range(1,6)],
            yticklabels=[f'ESI {i}' for i in range(1,6)])
axes[1].set_title(f'Confusion Matrix (QWK={meta_qwk:.4f})\nClinical cost: {total_cost:.4f}/patient', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('cost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total clinical cost:  {total_cost:.5f} per patient")
print(f"Undertriage cost:     {ut_cost:.5f}")
print(f"Overtriage cost:      {ot_cost:.5f}")
print(f"Undertriage/Total:    {ut_cost/total_cost*100:.1f}%")

In [ ]:
oof_df = X_main.copy()
oof_df['true'] = y_main
oof_df['pred'] = meta_preds
oof_df['undertriage'] = (oof_df['pred'] > oof_df['true']).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

bias_cols = [('sex', 'Sex'), ('insurance_type', 'Insurance'),
             ('language', 'Language'), ('age_group', 'Age Group')]

for i, (col, label) in enumerate(bias_cols):
    if col not in encoders:
        axes[i].text(0.5, 0.5, f'{col} not available', ha='center', transform=axes[i].transAxes)
        continue
    le = encoders[col]
    decoded = le.inverse_transform(oof_df[col].values.astype(int))
    temp = pd.DataFrame({'group': decoded, 'ut': oof_df['undertriage'].values})
    rates = temp.groupby('group')['ut'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    rates = rates[rates['count'] >= 30]

    colors_bar = ['#d62728' if r > rates['mean'].mean() + 2*rates['mean'].std() else '#1f77b4'
                  for r in rates['mean']]
    axes[i].barh(rates.index, rates['mean'] * 100, color=colors_bar)
    axes[i].axvline(rates['mean'].mean() * 100, color='gray', linestyle='--', alpha=0.7)
    axes[i].set_xlabel('Undertriage Rate (%)')
    axes[i].set_title(f'Undertriage by {label}', fontweight='bold')

    chi2, pval = stats.chi2_contingency(pd.crosstab(decoded, oof_df['undertriage'].values))[:2]
    axes[i].set_title(f'Undertriage by {label} (p={pval:.4f})', fontweight='bold')

plt.suptitle('Demographic Bias Audit — Systematic Undertriage Check', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('bias_audit.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
disagreements = oof_df[oof_df['pred'] < oof_df['true']].copy()  # Model says MORE urgent
disagreements['urgency_gap'] = disagreements['true'] - disagreements['pred']
disagreements['model_confidence'] = meta_oof[disagreements.index].max(axis=1)

print(f"TRIAGE DISAGREEMENT DETECTOR")
print(f"{'='*55}")
print(f"Cases where model suggests HIGHER acuity than assigned:")
print(f"  Total flagged: {len(disagreements):,} ({len(disagreements)/len(oof_df)*100:.1f}%)")
print(f"  Mean urgency gap: {disagreements['urgency_gap'].mean():.2f} ESI levels")
print(f"\n  Gap distribution:")
for gap in sorted(disagreements['urgency_gap'].unique()):
    n = (disagreements['urgency_gap'] == gap).sum()
    print(f"    +{int(gap)} ESI level(s) more urgent: {n:,} cases")

# Show top examples with high confidence
top_flags = disagreements.nlargest(5, 'model_confidence')
print(f"\nTop 5 highest-confidence disagreements (model most sure nurse underestimated):")
for idx, row in top_flags.iterrows():
    print(f"  Patient: assigned ESI {int(row['true'])} → model suggests ESI {int(row['pred'])} "
          f"(confidence: {row['model_confidence']:.3f})")

In [ ]:
from IPython.display import HTML

prototype_html = """
<div style="font-family: Arial, sans-serif; max-width: 700px; margin: 20px auto;
            border: 2px solid #2c3e50; border-radius: 12px; overflow: hidden;
            box-shadow: 0 4px 15px rgba(0,0,0,0.15);">

  <div style="background: linear-gradient(135deg, #2c3e50, #3498db); color: white;
              padding: 18px 24px;">
    <h2 style="margin: 0; font-size: 20px;">TriageGeist Clinical Decision Support</h2>
    <p style="margin: 4px 0 0 0; opacity: 0.85; font-size: 13px;">
      AI-Assisted Triage | Laitinen-Fredriksson Foundation
    </p>
  </div>

  <div style="padding: 20px 24px;">
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;">
      <div style="background: #f8f9fa; padding: 12px; border-radius: 8px;">
        <div style="font-size: 11px; color: #666; text-transform: uppercase; letter-spacing: 0.5px;">Patient</div>
        <div style="font-size: 16px; font-weight: bold; margin-top: 4px;">72F, Ambulance</div>
      </div>
      <div style="background: #f8f9fa; padding: 12px; border-radius: 8px;">
        <div style="font-size: 11px; color: #666; text-transform: uppercase; letter-spacing: 0.5px;">Chief Complaint</div>
        <div style="font-size: 14px; font-weight: bold; margin-top: 4px;">"Chest pain, difficulty breathing, nauseous"</div>
      </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin-bottom: 16px;">
      <div style="text-align: center; background: #fff3cd; padding: 8px; border-radius: 6px;">
        <div style="font-size: 10px; color: #856404;">HR</div>
        <div style="font-size: 18px; font-weight: bold; color: #856404;">112</div>
      </div>
      <div style="text-align: center; background: #f8d7da; padding: 8px; border-radius: 6px;">
        <div style="font-size: 10px; color: #721c24;">SpO2</div>
        <div style="font-size: 18px; font-weight: bold; color: #721c24;">91%</div>
      </div>
      <div style="text-align: center; background: #fff3cd; padding: 8px; border-radius: 6px;">
        <div style="font-size: 10px; color: #856404;">BP</div>
        <div style="font-size: 18px; font-weight: bold; color: #856404;">88/54</div>
      </div>
      <div style="text-align: center; background: #d4edda; padding: 8px; border-radius: 6px;">
        <div style="font-size: 10px; color: #155724;">Temp</div>
        <div style="font-size: 18px; font-weight: bold; color: #155724;">37.1°C</div>
      </div>
      <div style="text-align: center; background: #fff3cd; padding: 8px; border-radius: 6px;">
        <div style="font-size: 10px; color: #856404;">RR</div>
        <div style="font-size: 18px; font-weight: bold; color: #856404;">24</div>
      </div>
      <div style="text-align: center; background: #f8d7da; padding: 8px; border-radius: 6px;">
        <div style="font-size: 10px; color: #721c24;">GCS</div>
        <div style="font-size: 18px; font-weight: bold; color: #721c24;">13</div>
      </div>
    </div>

    <div style="background: #dc3545; color: white; padding: 16px; border-radius: 8px;
                margin-bottom: 12px; text-align: center;">
      <div style="font-size: 12px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.9;">
        AI Recommendation
      </div>
      <div style="font-size: 36px; font-weight: bold; margin: 6px 0;">ESI 2 — EMERGENT</div>
      <div style="font-size: 13px; opacity: 0.9;">
        Confidence: 94.2% | Conformal set: {ESI 1, ESI 2}
      </div>
    </div>

    <div style="background: #fff3cd; border-left: 4px solid #ffc107; padding: 12px;
                border-radius: 0 8px 8px 0; margin-bottom: 12px;">
      <div style="font-weight: bold; color: #856404; font-size: 13px;">
        ⚠ DISAGREEMENT ALERT — Nurse assigned ESI 3
      </div>
      <div style="font-size: 12px; color: #856404; margin-top: 6px;">
        Model suggests 1 level more urgent. Key drivers: hypoxia (SpO2 91%),
        hypotension (SBP 88), tachypnea (RR 24), chest pain + dyspnea keywords,
        elderly patient with cardiac history.
      </div>
    </div>

    <div style="background: #e9ecef; padding: 12px; border-radius: 8px;">
      <div style="font-size: 11px; color: #666; font-weight: bold; margin-bottom: 6px;">
        TOP SHAP DRIVERS FOR THIS PATIENT
      </div>
      <div style="font-size: 12px;">
        <span style="color: #dc3545;">▲ SpO2=91% (+0.42)</span> ·
        <span style="color: #dc3545;">▲ SBP=88 (+0.38)</span> ·
        <span style="color: #dc3545;">▲ "chest pain" (+0.31)</span> ·
        <span style="color: #dc3545;">▲ GCS=13 (+0.24)</span> ·
        <span style="color: #fd7e14;">▲ Age=72 (+0.18)</span> ·
        <span style="color: #fd7e14;">▲ hx_CAD (+0.15)</span>
      </div>
    </div>

    <div style="margin-top: 12px; font-size: 10px; color: #999; text-align: center;">
      This is a research prototype. All clinical decisions must be made by qualified clinicians.
      <br>Laitinen-Fredriksson Foundation · TriageGeist 2026
    </div>
  </div>
</div>
"""

display(HTML(prototype_html))
print("\nInteractive prototype rendered above.")
print("In deployment, this would be connected to the live model and EHR system.")

In [ ]:
test_preds = np.argmax(meta_test, axis=1) + 1

submission = pd.read_csv(PATH + 'sample_submission.csv')
submission['triage_acuity'] = test_preds
submission.to_csv('submission.csv', index=False)

print(f"FINAL RESULTS")
print(f"{'='*55}")
print(f"LightGBM OOF QWK:     {lgb_oof_qwk:.4f}")
print(f"XGBoost OOF QWK:      {xgb_oof_qwk:.4f}")
print(f"Stacked Meta QWK:     {meta_qwk:.4f}")
print(f"Stacked Accuracy:     {meta_acc:.4f}")
print(f"Clinical cost/patient: {total_cost:.5f}")
print(f"\nNovel contributions:")
print(f"  1. Clinical NLP embeddings: {'YES' if USE_EMBEDDINGS else 'TF-IDF only'}")
print(f"  2. Cost-sensitive training with asymmetric sample weights")
print(f"  3. Conformal prediction (coverage={coverage:.3f})")
print(f"  4. Triage Disagreement Detector ({len(disagreements):,} flags)")
print(f"  5. Interactive HTML clinical prototype")
print(f"\nTest prediction distribution:")
for esi in range(1, 6):
    n = (test_preds == esi).sum()
    print(f"  ESI {esi}: {n:,} ({n/len(test_preds)*100:.1f}%)")
print(f"\nSubmission saved: submission.csv ({len(submission):,} rows)")